# Wristband repulsion — same quality, less time

One network, one dataset, three ways to compute the repulsion term. The question is whether the two
cheap paths reach the quality of the exact one, and how much time that saves at a large batch.

| path | what it does | cost |
|---|---|---|
| `pairwise` | the explicit 3-image kernel | `O(N^2 d)` |
| `spectral` | the same series cut at spherical-harmonic degree 1 | `O(N d K)` |
| `poisson` | unbiased random features in place of the cut | `O(N d D c)` |

**Nothing here is measured by this notebook.** Every number comes from a test that already lives in
`ml-tidbits/python/tests/`, called with its own keyword arguments and charted from what it returns.
That is deliberate: a benchmark that re-implements the thing it measures drifts away from it.

Two charts.

1. **Figure 1** — `TestPoissonTrainingRun`. Trains `DeterministicGaussianAutoencoder` once per path
   and scores the held-out latents. Where each path lands, and what it cost per epoch.
2. **Figure 2** — `TestPoissonTiming`. Milliseconds per call against batch size, which is where the
   `O(N^2)` path turns over.

Then one printed table, `TestPoissonVarianceAndD`, which is where the feature budget `D` comes from.

### Why the training comparison is like for like

This is the part that makes or breaks it, and the repo already handles it.

- `torch.manual_seed(seed)` inside `_TrainOne` fixes the initial weights **and** the batch
  permutation, so every path sees identical weights and identical batches.
- **Shared calibration.** `C_WristbandGaussianLoss` z-scores its components by constants fixed when
  it is built. Three paths built independently therefore descend three differently scaled
  objectives — and a sampled estimator inflates `std_total`, so it would be handed a quieter loss.
  The test builds all three, then copies `pairwise`'s constants onto the others, so they descend one
  scale. It also reports the uncalibrated mode for contrast.
- **The judge grades nobody's own work.** It is a separate uncalibrated exact-`pairwise` loss applied
  to a **held-out** split under `no_grad`, alongside `W2ToStandardNormalSq(z) / d`. No path trains
  against it.
- `target` is the judge's value on a genuine `N(0, I)` batch of the same shape. That is the floor the
  bars are read against, drawn as a dashed line.

`pairwise` is trained and charted alongside the other two, never assumed. It is also the objective
the other two approximate, so a feature path *beating* it on the judge is something to investigate,
not a win.


## 0. Setup

`PRESET = "smoke"` is the repo's own sizes with fewer seeds and epochs, quick enough to check the
notebook works. `PRESET = "full"` raises the batch and the dataset for the large-`N` claim, and
wants a GPU.

**Figure 1 is the expensive cell, and its cost is not obvious.** Each seed buys three whole
trainings, and the test adds a fourth set for its own-calibration mode, so the run is
`3 x (seeds + 1)` trainings — nine at two seeds, eighteen at five. Budget accordingly before
raising `seeds`. Nothing is written to the cache until that cell finishes, so interrupting it
loses the whole cell rather than the current run.

Results are cached to `results/`, so re-running a cell after its measurement redraws instantly.


In [ ]:
# --- settings ---------------------------------------------------------------------------------
PRESET          = "smoke"   # "smoke" (a few minutes) or "full" (large batch, wants a GPU)
FORCE_RECOMPUTE = False     # True re-runs every measurement and overwrites the cache

# Where the ml-tidbits code comes from. Setting ML_TIDBITS to a path overrides everything else,
# which is how you would point at a Google Drive mount. Otherwise the notebook looks on disk and,
# failing that, clones. The URL is the fork: the original repo carries no poisson path.
ML_TIDBITS        = None
ML_TIDBITS_REPO   = "https://github.com/andremiguelc/ml-tidbits"
ML_TIDBITS_BRANCH = "poisson-mode-sampling"

# Only the tests' own keyword arguments appear here. Everything that defines the experiment --
# the model, the data generator, lr, the loss weights, k_modes, embed_dim, in_dim, the judge --
# stays at the value the repo uses.
SIZES = {
   "smoke": dict(
      batch_size   = 1024,                        # the N the repulsion sees, per step
      n_features   = 384,                         # D
      n_samples    = 20_000,
      n_epochs     = 4,
      seeds        = (0, 1),
      cal_reps     = 64,                          # see the note below on where these go
      time_sizes   = (256, 1024, 4096),
      var_features = (32, 128, 384, 1024, 4096),  # the D sweep behind the table at the end
   ),
   "full": dict(
      batch_size   = 8192,                        # above the crossover: this one is the point
      n_features   = 1024,
      n_samples    = 200_000,                     # 19 steps an epoch, so training still converges
      n_epochs     = 12,
      seeds        = (0, 1),                      # each seed costs three whole trainings
      cal_reps     = 32,
      time_sizes   = (1024, 4096, 8192),          # 16384 needs ~8 GiB for the pairwise kernel
      var_features = (32, 128, 384, 1024, 4096),
   ),
   # Where the time goes, and why these values. TestPoissonTrainingRun runs two modes: `shared
   # calibration` over every seed, and `own calibration` over the first seed only. That is
   # 3 paths * (len(seeds) + 1) whole trainings -- eighteen of them at five seeds, not three.
   # Separately, it rebuilds a loss inside the innermost loop, and each build spends `cal_reps`
   # forward passes measuring calibration constants. In the shared mode the next line overwrites
   # those constants with pairwise's, so that work is discarded; only the `own calibration` runs
   # and the three reference objects keep theirs. Hence a low `cal_reps` and few seeds, and hence
   # `batch_size` and `n_samples` left alone -- they are what the result is about.
}
CFG = SIZES[PRESET]

PATHS = ("pairwise", "spectral", "poisson")
TIME_D = 128                # the embedding dimension the repo's timing test uses

# Below roughly this batch the sampler costs more per call than the exact kernel, so no speed
# comparison there can come out above 1.0x. Measured at 4096 by the repo's own timing test at
# d=128, D=384; figure 2 measures it again for whatever settings this preset uses.
CROSSOVER_HINT = 4096

print(f"preset {PRESET}")
for key, value in CFG.items():
   print(f"   {key:<13} {value}")


In [ ]:
import importlib.machinery, importlib.util
import json, subprocess, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt


def _FindOnDisk(start: Path):
   """`ml-tidbits/python` at or above `start`, or None. The local route: no network, no clone."""
   for p in [start, *start.parents]:
      if (p / "ml-tidbits" / "python").is_dir():
         return p / "ml-tidbits" / "python"
   return None


def _Clone(into: Path) -> Path:
   """Clone the branch carrying the poisson path. The Colab route. An existing clone is reused."""
   target = into / "ml-tidbits"
   if not (target / "python").is_dir():
      subprocess.run(["git", "clone", "--depth", "1", "--branch", ML_TIDBITS_BRANCH,
                      ML_TIDBITS_REPO, str(target)], check=True)
   return target / "python"


def _HeadSha(repo_dir: Path) -> str:
   """Short commit of the checkout in use, so a run can be traced back to code."""
   try:
      done = subprocess.run(["git", "-C", str(repo_dir), "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True, check=True)
      return done.stdout.strip()
   except Exception:
      return "unknown"


_CWD = Path.cwd().resolve()
_ON_DISK = _FindOnDisk(_CWD)
if ML_TIDBITS:
   ML_PATH, _ROUTE = Path(ML_TIDBITS).expanduser(), "set by hand"
elif _ON_DISK is not None:
   ML_PATH, _ROUTE = _ON_DISK, "found on disk"
else:
   ML_PATH, _ROUTE = _Clone(_CWD), f"cloned {ML_TIDBITS_REPO} at {ML_TIDBITS_BRANCH}"
if str(ML_PATH) not in sys.path:
   sys.path.insert(0, str(ML_PATH))


def _BindTestsPackage(tests_dir: Path) -> None:
   """Point the name `tests` at this checkout, whatever else on the machine answers to it.

   Neither `tests` nor `embed_models` holds an __init__.py, so both are namespace packages. A
   regular package of the same name anywhere on sys.path beats a namespace portion no matter the
   order, and `tests` is a common enough name that a hosted runtime supplies one. Nothing is
   called `embed_models`, which is why only this half needs help. The helpers import each other
   as `tests.X`, so a plain by-path load would not be enough.
   """
   for name in [m for m in list(sys.modules) if m == "tests" or m.startswith("tests.")]:
      del sys.modules[name]
   spec = importlib.machinery.ModuleSpec("tests", None, is_package=True)
   spec.submodule_search_locations = [str(tests_dir)]
   sys.modules["tests"] = importlib.util.module_from_spec(spec)


_BindTestsPackage(ML_PATH / "tests")

from embed_models.EmbedModels import C_WristbandGaussianLoss
from tests.TestPoissonTraining import TestPoissonTrainingRun
from tests.TestPoissonWristband import TestPoissonTiming, TestPoissonVarianceAndD

# Fail here with an instruction, rather than deep inside a figure. The original ml-tidbits repo
# has no poisson path at all, so aiming this at the wrong repo or branch is an easy mistake.
try:
   C_WristbandGaussianLoss(repulsion="poisson", reduction="global", calibration_shape=None)
except (TypeError, ValueError) as _err:
   raise RuntimeError(
      f"this checkout of ml-tidbits has no poisson path ({_err}). Point ML_TIDBITS_BRANCH at the "
      "branch that carries it, or ML_TIDBITS at a checkout that does.") from _err

NB_DIR  = Path.cwd()
FIG_DIR = NB_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
RES_DIR = NB_DIR / "results"; RES_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- style ------------------------------------------------------------------------------------
# One colour per path, the same in both figures. Grey is the exact path, so it stays neutral.
COLOR = {"pairwise": "#8a8a8a", "spectral": "#e08214", "poisson": "#2a5d9f"}

mpl.rcParams.update({
   "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
   "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
   "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.6,
   "axes.spines.top": False, "axes.spines.right": False,
   "axes.axisbelow": True,
   "legend.frameon": False, "legend.fontsize": 9,
   "lines.linewidth": 2.0, "lines.markersize": 6,
   "figure.facecolor": "white", "axes.facecolor": "white",
})


# --- cache --------------------------------------------------------------------------------------

def _Jsonable(o):
   if hasattr(o, "tolist"):
      return o.tolist()
   if isinstance(o, (np.floating, np.integer)):
      return o.item()
   raise TypeError(f"cannot store {type(o)} in the cache")


def cached(name, fn):
   """Run `fn` once, keep the result on disk, and skip it on every later run."""
   path = RES_DIR / f"{PRESET}__{name}.json"
   if path.exists() and not FORCE_RECOMPUTE:
      print(f"cache hit   {path.name}   (delete it, or set FORCE_RECOMPUTE, to measure again)\n")
      return json.loads(path.read_text())
   start = time.perf_counter()
   out = fn()
   path.write_text(json.dumps(out, indent=1, default=_Jsonable))
   print(f"\nmeasured    {path.name}  in {time.perf_counter() - start:.1f} s")
   return json.loads(path.read_text())   # round-trip, so a cache hit and a fresh run agree


def _IntKeys(blob):
   """JSON turns integer keys into strings. Put them back."""
   return {int(k): v for k, v in blob.items()}


print(f"{'ml-tidbits':<11} {ML_PATH}")
print(f"{'source':<11} {_ROUTE}, commit {_HeadSha(ML_PATH.parent)}")
print(f"{'figures':<11} {FIG_DIR}")
print(f"{'device':<11} {DEVICE}"
      + (f"  ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
if DEVICE.type == "cuda":
   # Restarting this runtime does not free memory held by a *different* session on the same card,
   # which is the usual reason this number stays low after a restart. `!nvidia-smi` lists the
   # processes; Colab's Runtime > Manage sessions is where the other ones get terminated.
   _free, _total = torch.cuda.mem_get_info()
   print(f"{'gpu memory':<11} {_free / 2 ** 30:.1f} GiB free of {_total / 2 ** 30:.1f} GiB")
   if _free < 0.6 * _total:
      print(f"{'':11} another session is holding this card. Run !nvidia-smi to see which, then")
      print(f"{'':11} Runtime > Manage sessions to end it. Restarting this one will not do it.")


## Figure 1 — same quality, less time

`TestPoissonTrainingRun` trains the same autoencoder once per path: the `C_EmbedAttentionModule`
encoder, the invertible flow, and the decoder, on a five-cluster non-Gaussian mixture. The loss is
`lambda_rec * reconstruction + lambda_wb * wristband`, and the only thing that differs between runs
is which repulsion the wristband term uses.

**(a)** where each path landed — the judge's exact-`pairwise` repulsion on the held-out split, lower
is better, with the dashed line at `target`, the judge's value on a real `N(0, I)` batch of the same
shape. Error bars are the spread over seeds.

**(b)** what that cost, in seconds per epoch.

Read them together. If (a) is flat and (b) is not, that is the result. If (a) is *not* flat, the
paths did not reach the same quality and (b) is not yet a speed result — the table at the end says
which way to move `D`.

The test's own output prints first, including the uncalibrated mode and the paired seed-by-seed
differences, which cancel seed-to-seed variation.


In [ ]:
# The pairwise path holds several batch-by-batch float32 tensors through the backward pass. That
# is what decides whether this cell fits, so price it before spending twenty minutes finding out.
def _PairGiB(batch):
   """Roughly what the pairwise kernel alone costs at this batch. Memory goes as the square."""
   return 8. * 4. * float(batch) ** 2 / 2. ** 30


print(f"pairwise needs about {_PairGiB(CFG['batch_size']):.1f} GiB for its {CFG['batch_size']}-by-"
      f"{CFG['batch_size']} kernel, before the model's own activations")
if DEVICE.type == "cuda":
   _free_now = torch.cuda.mem_get_info()[0] / 2. ** 30
   print(f"the card has {_free_now:.1f} GiB free")
   if _free_now < 2.5 * _PairGiB(CFG["batch_size"]):
      print("this is tight. Free the card first -- see the note in the setup cell -- or lower")
      print("batch_size in the settings cell. Memory goes as the square, so:")
      for _b in (8192, 6144, 4096):
         print(f"   batch {_b:>5}  kernel {_PairGiB(_b):>4.1f} GiB"
               + ("   (at the crossover; below this there is no speedup to measure)"
                  if _b <= CROSSOVER_HINT else ""))


def _RunTraining():
   return TestPoissonTrainingRun(
      batch_size=int(CFG["batch_size"]),
      n_features=int(CFG["n_features"]),
      n_samples=int(CFG["n_samples"]),
      n_epochs=int(CFG["n_epochs"]),
      seeds=tuple(CFG["seeds"]),
      calibration_reps=int(CFG["cal_reps"]),
      paths=PATHS,                    # named, so pairwise is trained rather than assumed
   )


TRAIN = cached("training", _RunTraining)
TRAIN_MODE = "shared calibration"     # the fair one: every path on pairwise's loss scale
T_ROWS = TRAIN["results"][TRAIN_MODE]
T_PATHS = [p for p in PATHS if p in T_ROWS]


def _Spread(path, key):
   """Mean and standard deviation of one metric across the seeds of that path."""
   vals = [r[key] for r in T_ROWS[path]["runs"]]
   return float(np.mean(vals)), (float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.)


print(f"preset {PRESET}: {TRAIN_MODE}, batch={CFG['batch_size']}, D={CFG['n_features']}, "
      f"{CFG['n_epochs']} epochs, {len(CFG['seeds'])} seeds")
print(f"the judge on a true N(0,I) batch: {TRAIN['target']:.6f}   (lower is better)")
if int(CFG["batch_size"]) < CROSSOVER_HINT:
   print(f"NOTE  this batch is below the crossover near N={CROSSOVER_HINT}, where the sampler "
         f"first becomes\n      cheaper than the exact kernel. Panel (b) will read about 1.0x, "
         f"and that is the setting\n      talking, not the method. Use PRESET = \"full\" for the "
         f"speed comparison.")
print()
print(f"{'path':<10} {'s/epoch':>9} {'speedup':>9} {'recon MSE':>12} {'judge':>12} {'W2/d':>12}")
for path in T_PATHS:
   secs = T_ROWS[path]["seconds"]
   up = T_ROWS["pairwise"]["seconds"] / secs if "pairwise" in T_ROWS and secs else None
   print(f"{path:<10} {secs:>9.2f} " + (f"{up:>8.2f}x" if up else f"{'--':>9}")
         + f" {T_ROWS[path]['mse']:>12.5f} {T_ROWS[path]['judge_rep']:>12.6f} "
           f"{T_ROWS[path]['judge_w2']:>12.5f}")

# --- figure -------------------------------------------------------------------------------------
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(11.0, 4.4))
colours = [COLOR[p] for p in T_PATHS]

judge = [_Spread(p, "judge_rep")[0] for p in T_PATHS]
judge_sd = [_Spread(p, "judge_rep")[1] for p in T_PATHS]
bars = ax_a.bar(T_PATHS, judge, width=0.55, color=colours, yerr=judge_sd,
                error_kw=dict(ecolor="black", lw=1.0, capsize=4))
ax_a.bar_label(bars, fmt="%.4f", fontsize=9, padding=4)
ax_a.axhline(TRAIN["target"], color="black", lw=1.2, ls="--")
ax_a.text(0.99, TRAIN["target"], " a true N(0,I) batch", transform=ax_a.get_yaxis_transform(),
          ha="right", va="bottom", fontsize=8.5)
ax_a.set_ylabel("judge: exact pairwise repulsion, held out")
ax_a.set_title("(a) where each path landed   (lower is better)")

secs = [T_ROWS[p]["seconds"] for p in T_PATHS]
bars = ax_b.bar(T_PATHS, secs, width=0.55, color=colours)
ax_b.bar_label(bars, fmt="%.2f s", fontsize=9, padding=4)
if "pairwise" in T_ROWS:
   for rect, path, value in zip(bars, T_PATHS, secs):
      if path == "pairwise" or not value:
         continue
      ax_b.annotate(f"{T_ROWS['pairwise']['seconds'] / value:.1f}x faster",
                    xy=(rect.get_x() + rect.get_width() / 2., value), xytext=(0, 26),
                    textcoords="offset points", ha="center", fontsize=10,
                    color=COLOR[path], fontweight="bold")
ax_b.set_ylim(0, max(secs) * 1.35)
ax_b.set_ylabel("seconds per epoch")
ax_b.set_title(f"(b) what it cost   (preset {PRESET}: batch {CFG['batch_size']}, "
               f"D {CFG['n_features']})")
if int(CFG["batch_size"]) < CROSSOVER_HINT:
   ax_b.text(0.5, 0.93, f"batch is below the N~{CROSSOVER_HINT} crossover:\nno speedup is "
                        f"available here", transform=ax_b.transAxes, ha="center", va="top",
             fontsize=9, color="#b2182b")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_training.png")
plt.show()


## Figure 2 — kernel cost against batch size

`TestPoissonTiming`, forward and backward together, median of five calls, at `d = 128` and the same
`D` figure 1 trained with. Both axes are logarithmic, so a straight line is a power law.

`pairwise` builds an `N` by `N` matrix, so its line is the steep one. The crossover marked on the
chart is the batch size above which the feature paths are cheaper — below it, the exact path is fine
and there is nothing to buy.

The test also times a `flip_pool = 64` variant of the sampler. That is in its printed table rather
than on the chart.


In [ ]:
def _RunTiming():
   """The repo's timer does not catch out-of-memory, so drop the largest N and retry if it fails."""
   sizes = list(CFG["time_sizes"])
   while sizes:
      try:
         return TestPoissonTiming(d=TIME_D, sizes=tuple(sizes),
                                  n_features=int(CFG["n_features"]))
      except RuntimeError as err:
         if "out of memory" not in str(err).lower():
            raise
         if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
         print(f"\n   N={sizes.pop()} did not fit in memory; retrying without it\n", flush=True)
   raise RuntimeError("not even the smallest batch fitted; restart the runtime")


TIMING = cached("timing", _RunTiming)
T_BOTH = _IntKeys(TIMING["forward_backward"])
T_NS = sorted(T_BOTH)

# The test reports its crossover on the forward-only timings. Training pays for the backward too,
# so the chart plots forward+backward and takes its marker from that same series.
T_CROSS = next((n for n in T_NS if T_BOTH[n]["poisson"] < T_BOTH[n]["pairwise"]), None)

print(f"forward+backward, d={TIME_D}, D={CFG['n_features']}, {TIMING['device']}, milliseconds")
print(f"{'N':>7} " + " ".join(f"{p:>12}" for p in PATHS) + f" {'pair/pois':>11}")
for n in T_NS:
   row = T_BOTH[n]
   up = row["pairwise"] / row["poisson"] if row.get("poisson") else None
   print(f"{n:>7} " + " ".join(f"{row[p]:>12.2f}" for p in PATHS)
         + (f" {up:>10.2f}x" if up else f" {'--':>11}"))
print("\npoisson passes pairwise at N = "
      + (str(T_CROSS) if T_CROSS else f"above {T_NS[-1]}") + " on forward+backward, and at N = "
      + (str(TIMING["crossover"]) if TIMING["crossover"] else f"above {T_NS[-1]}")
      + " on the forward pass alone")

# --- figure -------------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8.0, 4.4))
for path in PATHS:
   ax.plot(T_NS, [T_BOTH[n][path] for n in T_NS], marker="o", color=COLOR[path], label=path)
if T_CROSS:
   ax.axvline(T_CROSS, color="#b2182b", lw=1.2, ls="--")
   ax.text(T_CROSS * 1.1, 0.04, f"crossover\nN = {T_CROSS}",
           transform=ax.get_xaxis_transform(), fontsize=8.5, color="#b2182b")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xticks(T_NS); ax.set_xticklabels([str(n) for n in T_NS])
ax.minorticks_off()
ax.set_xlabel("batch size N")
ax.set_ylabel("milliseconds per call")
ax.set_title(f"forward and backward, d={TIME_D}, D={CFG['n_features']}, {TIMING['device']}")
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_timing.png")
plt.show()


## Where the feature budget comes from

`TestPoissonVarianceAndD`, printed only. It sweeps `D` on a clustered batch and reports the relative
spread of the energy against its `sqrt(exp(2c) / D)` bound, the gradient cosine against the exact
target, and the measured bias the logarithm introduces against its prediction.

This is the table to consult when figure 1(a) is not flat: the gradient cosine column says whether
`D` is large enough for training, which is a stiffer requirement than a single call.

It is not cached, so the numbers always print. The repo asserts inside it that the spread stays under
the bound and that the gradient improves with `D`; if an assertion trips it is reported rather than
allowed to stop the notebook.


In [ ]:
try:
   TestPoissonVarianceAndD(features=tuple(CFG["var_features"]))
except AssertionError as _err:
   print(f"\nan assertion inside the repo's test did not hold: {_err}")


## The numbers, in one block

Figures land in `figures/`, measurements in `results/<preset>__<name>.json`. Delete one result file
to redo one measurement; set `FORCE_RECOMPUTE = True` for one pass to redo them all. The preset is
part of the file name, so a smoke run and a full run do not overwrite each other.


In [ ]:
print(f"preset {PRESET}   device {TIMING['device']}   batch {CFG['batch_size']}   "
      f"D {CFG['n_features']}   {CFG['n_epochs']} epochs   {len(CFG['seeds'])} seeds")
print("=" * 88)

print(f"\nTRAINING   ({TRAIN_MODE}, held-out judge, lower is better)")
print(f"   a true N(0,I) batch scores {TRAIN['target']:.6f}")
for path in T_PATHS:
   mean, sd = _Spread(path, "judge_rep")
   print(f"   {path:<9} judge {mean:>10.6f} +- {sd:.6f}   "
         f"MSE {T_ROWS[path]['mse']:>8.5f}   {T_ROWS[path]['seconds']:>7.2f} s/epoch")
if "pairwise" in T_ROWS:
   for path in T_PATHS:
      if path != "pairwise" and T_ROWS[path]["seconds"]:
         print(f"   {path:<9} is {T_ROWS['pairwise']['seconds'] / T_ROWS[path]['seconds']:.2f}x "
               f"faster per epoch than pairwise")

print(f"\nKERNEL COST   (forward+backward, d={TIME_D})")
_big = T_NS[-1]
print(f"   at N={_big}: " + ",   ".join(f"{p} {T_BOTH[_big][p]:.2f} ms" for p in PATHS))
print("   poisson passes pairwise at N = " + (str(T_CROSS) if T_CROSS else f"above {T_NS[-1]}"))
print("=" * 88)
